## How to read Tree files (`.valtree.root` files)

In [ ]:
import ROOT
file = "/eos/user/o/oarquero/TB2026CERN/data/TB2026CERN_run_000007/ecal_TB2026CERN_run_000007.valtree.root"
f = ROOT.TFile.Open(file)

In [ ]:
f.ls()
t = f.Get("ecal")
# All the information about the branches can be found in the README.md
t.Print()

In [ ]:
# With RDataFrame is easier to explore
df = ROOT.RDataFrame("ecal", file)
print(df.GetColumnNames())
h = df.Histo1D("nhit")
c = ROOT.TCanvas()
h.Draw()
c.Draw()

In [ ]:
# With RDataFrame is easier to explore
df = ROOT.RDataFrame("ecal", file)
print(df.GetColumnNames())
h = df.Histo1D("energy")
c = ROOT.TCanvas()
h.Draw()
c.Draw()

In [ ]:
# Get event index (is an special code, not 0,1,...)
# Select an event wich is probably an electron
events = df.Filter("mip_likeness < 0.05").AsNumpy(["event"])
# Other variables could be used for filtering, i.e. nhit

In [ ]:
event = events["event"][0]
data_0 = df.Filter(f"event == {event}").AsNumpy()

# Get coordinates
hit_x = data_0["hit_x"][0]
hit_y = data_0["hit_y"][0]
hit_z = data_0["hit_z"][0]

In [ ]:
# Plot projections
import matplotlib.pyplot as plt
ax = plt.figure().add_subplot(projection='3d')

ax.scatter(hit_z, hit_x, hit_y)
ax.set(
    xlabel='Z [mm]',
    ylabel='X [mm]',
    zlabel='Y [mm]',

    xlim=(0,-225),
    ylim=(-86.30,86.30),
    zlim=(-86.30,86.30)
    
)
ax.view_init(40, -120, 0)
plt.show()

## How to read Tree files (`.edm4hep.root` files)

This format is easier to use with edm4hep pipelines, with more difficult to explore data, as high level features are stored in a more "complicated" way (see README.md).

In [ ]:
# For edm4hep, we use Podio
from podio.root_io import Reader
edm4hep_events = "/eos/user/o/oarquero/TB2026CERN/data/TB2026CERN_run_000048/ecal_TB2026CERN_run_000048.edm4hep.root"
reader = Reader(edm4hep_events)
print(reader.categories)

In [ ]:
for event in reader.get("events"):
  hits = event.get("ECalHits")
  print("Number of hits:", len(hits))
  for hit in hits:
    p = hit.getPosition()
    print(f"x :{round(p.x, 2)}, y {round(p.y, 2)}, z {round(p.z, 2)}, Energy {round(hit.getEnergy(), 3)}")
  break

In [ ]:
# Get one event
import numpy as np
from podio.root_io import Reader

events = Reader(edm4hep_events).get("events")
fr = events[0]
hits = fr.get("ECalHits")            # edm4hep::CalorimeterHitCollection

energy = np.asarray(hits.energy())   # float32, shape (12,)
cellid = np.asarray(hits.cellID())   # uint64,  shape (12,)
